In [ ]:
!pip install transformers==4.37.2 accelerate==0.27.2 peft==0.8.2 datasets evaluate sentencepiece -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


model_path = "/kaggle/input/datasets/balancelott/acos-vit5-retrained" 

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang nạp mô hình lên {device}...")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

model.eval()
print("Sẵn sàng!")

## Test ví dụ mẫu

In [ ]:
import json

def predict_acos(review_text):
    input_text = f"review: {review_text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=256)
        
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    print("-" * 50)
    print(f" Câu review: {review_text}\n")
    print("Kết quả mô hình trích xuất:")
    

    try:
        parsed_json = json.loads(prediction)
        print(json.dumps(parsed_json, indent=4, ensure_ascii=False))
    except Exception:
        print("[Cảnh báo] Mô hình sinh ra chuỗi JSON không hợp lệ:")
        print(prediction)
    print("-" * 50)

predict_acos("Mình và bạn bè vừa ghé quán bia Thượng Hải, phải nói là rất ấn tượng. Không gian thoáng mát, sạch sẽ, nhân viên phục vụ nhanh nhẹn và nhiệt tình. Đồ ăn ở đây ngon, hợp khẩu vị, đặc biệt các món nhắm với bia rất chuẩn. Giá cả hợp lý so với chất lượng. Rất đáng để quay lại cùng bạn bè vào dịp cuối tuần!")

In [ ]:
import pandas as pd
from datasets import Dataset
data_path = "/kaggle/input/datasets/balancelott/acos-dataset/full_data_10k.jsonl"

processed_data = []

with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        item = json.loads(line.strip())
        review_text = item.get("review_text", "")
        
        if "annotations" in item and item["annotations"]:
            acos_list = []
            for ann in item["annotations"]:
                acos_list.append({
                    "aspect_expression": str(ann.get("aspect_expression", "NULL")).strip(),
                    "aspect_category": str(ann.get("aspect_category", "NULL")).strip(),
                    "sentiment": str(ann.get("sentiment", "NULL")).strip()
                })
            
            if acos_list:
                processed_data.append({
                    "input_text": f"review: {review_text}",
                    "target_text": json.dumps(acos_list, ensure_ascii=False)
                })

df = pd.DataFrame(processed_data)
dataset = Dataset.from_pandas(df)
test_dataset = dataset.train_test_split(test_size=0.2, seed=42)["test"]

print(f"Đã tái tạo thành công {len(test_dataset)} mẫu Test.")

print("\n=== SO SÁNH 5 MẪU ĐẦU TIÊN ===")
for i in range(5):
    sample = test_dataset[i]
    review = sample["input_text"].replace("review: ", "")
    ground_truth = sample["target_text"]
    
    inputs = tokenizer(sample["input_text"], return_tensors="pt", max_length=128, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=256)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    print(f"\n[Câu {i+1}]: {review}")
    print(f"Thực tế : {ground_truth}")
    print(f"Dự đoán : {prediction}")

In [ ]:
import json
import re
import torch
import numpy as np
from tqdm import tqdm

def extract_elements_salvaged(json_str, task_type):
    try:
        data = json.loads(json_str)
        if isinstance(data, list):
            elements = []
            for item in data:
                if not isinstance(item, dict): continue
                a = str(item.get("aspect_expression", "")).strip().lower()
                c = str(item.get("aspect_category", "")).strip().lower()
                s = str(item.get("sentiment", "")).strip().lower()
                
                if task_type == "aspect" and a and a != "null": elements.append(a)
                elif task_type == "category" and c and c != "null": elements.append(c)
                elif task_type == "sentiment" and s and s != "null": elements.append(s)
                elif task_type == "combined" and a and a != "null": elements.append((a, c, s))
            return elements
    except Exception:
        pass 

    pattern = r'"(aspect_expression|aspect_category|opinion_expression|sentiment)":\s*"([^"]+)"'
    matches = re.findall(pattern, json_str)
    
    salvaged_data = []
    current_item = {}
    
    for key, value in matches:
        current_item[key] = value
        if key == "sentiment" or (key == "aspect_expression" and len(current_item) > 1):
            if "sentiment" in current_item:
                salvaged_data.append(current_item.copy())
                current_item = {}
            elif key == "aspect_expression":
                temp_aspect = current_item["aspect_expression"]
                current_item = {"aspect_expression": temp_aspect}
                
    elements = []
    for item in salvaged_data:
        a = str(item.get("aspect_expression", "")).strip().lower()
        c = str(item.get("aspect_category", "")).strip().lower()
        s = str(item.get("sentiment", "")).strip().lower()
        
        if task_type == "aspect" and a and a != "null": elements.append(a)
        elif task_type == "category" and c and c != "null": elements.append(c)
        elif task_type == "sentiment" and s and s != "null": elements.append(s)
        elif task_type == "combined" and a and a != "null": elements.append((a, c, s))
    return elements

batch_size = 32  
predictions = []
references = test_dataset["target_text"]

print("Đang chạy dự đoán trên toàn bộ tập Test...")
for i in tqdm(range(0, len(test_dataset), batch_size)):
    batch = test_dataset[i : i + batch_size]
    
    inputs = tokenizer(
        batch["input_text"], 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=256)
        
    decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend([pred.strip() for pred in decoded_preds])

counters = {
    "aspect": {"tp": 0, "fp": 0, "fn": 0},
    "category": {"tp": 0, "fp": 0, "fn": 0},
    "sentiment": {"tp": 0, "fp": 0, "fn": 0},
    "combined": {"tp": 0, "fp": 0, "fn": 0}
}

for pred_str, label_str in zip(predictions, references):
    for task in counters.keys():
        pred_set = set(extract_elements_salvaged(pred_str, task))
        gold_set = set(extract_elements_salvaged(label_str, task))
        
        counters[task]["tp"] += len(pred_set & gold_set) 
        counters[task]["fp"] += len(pred_set - gold_set) 
        counters[task]["fn"] += len(gold_set - pred_set) 

print("\n" + "="*60)
print("BÁO CÁO ĐÁNH GIÁ MÔ HÌNH ACOS TRÊN TẬP KIỂM THỬ (TEST SET)")
print("="*60)

for task, counts in counters.items():
    tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    task_name = task.upper() if task != "combined" else "COMBINED (Strict - Đúng cả 3)"
    print(f" Tiêu chí: {task_name}")
    print(f"   - Độ chính xác (Precision) : {precision * 100:.2f}%")
    print(f"   - Độ bao phủ (Recall)      : {recall * 100:.2f}%")
    print(f"   - Điểm F1-Score            : {f1 * 100:.2f}%")
    print("-" * 45)

In [ ]:
import pandas as pd

target_sentiments = ["positive", "neutral", "negative"]
target_categories = [
    "Food Quality", "Food Safety", "Price", "Cleanliness", 
    "Ambience", "Location", "Menu", "Service"
]

sentiment_counters = {s: {"tp": 0, "fp": 0, "fn": 0} for s in target_sentiments}
category_counters = {c: {"tp": 0, "fp": 0, "fn": 0} for c in target_categories}

for pred_str, label_str in zip(predictions, references):
    pred_sentiments = set(extract_elements_salvaged(pred_str, "sentiment"))
    gold_sentiments = set(extract_elements_salvaged(label_str, "sentiment"))
    
    for s in target_sentiments:
        in_pred = s in pred_sentiments
        in_gold = s in gold_sentiments
        if in_pred and in_gold:
            sentiment_counters[s]["tp"] += 1
        elif in_pred and not in_gold:
            sentiment_counters[s]["fp"] += 1
        elif not in_pred and in_gold:
            sentiment_counters[s]["fn"] += 1

    pred_categories = set(extract_elements_salvaged(pred_str, "category"))
    gold_categories = set(extract_elements_salvaged(label_str, "category"))
    
    for c in target_categories:
        in_pred = c in pred_categories
        in_gold = c in gold_categories
        if in_pred and in_gold:
            category_counters[c]["tp"] += 1
        elif in_pred and not in_gold:
            category_counters[c]["fp"] += 1
        elif not in_pred and in_gold:
            category_counters[c]["fn"] += 1

def generate_report_df(counters, display_mapping, column_name):
    rows = []
    for key, counts in counters.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        rows.append({
            column_name: display_mapping.get(key, key),
            "Precision": f"{precision * 100:.2f}%",
            "Recall": f"{recall * 100:.2f}%",
            "F1-score": f"{f1 * 100:.2f}%"
        })
    return pd.DataFrame(rows)

sentiment_display = {"positive": "Positive", "neutral": "Neutral", "negative": "Negative"}
category_display = {c: c.title() for c in target_categories}

df_sentiment = generate_report_df(sentiment_counters, sentiment_display, "Sentiment")
df_category = generate_report_df(category_counters, category_display, "Category")

print("\n" + "="*50)
print("4. BẢNG KẾT QUẢ THEO TỪNG SENTIMENT")
print("="*50)
display(df_sentiment)

print("\n" + "="*50)
print("5. BẢNG KẾT QUẢ THEO TỪNG ASPECT CATEGORY")
print("="*50)
display(df_category)

In [ ]:
import pandas as pd
from collections import Counter

def count_categories(dataset_path):
    cat_counts = Counter()
    total_samples = 0
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line.strip())
            if "annotations" in item:
                # Dùng set để chỉ đếm mỗi category 1 lần trên 1 câu review (nếu muốn đếm tổng số annotation thì bỏ set)
                categories_in_sample = {ann.get("aspect_category") for ann in item["annotations"]}
                cat_counts.update(categories_in_sample)
                total_samples += 1
    return cat_counts, total_samples

data_path = "/kaggle/input/datasets/balancelott/acos-dataset/full_data_10k.jsonl"

full_dataset = Dataset.from_pandas(pd.DataFrame([json.loads(line) for line in open(data_path, 'r', encoding='utf-8')]))
splits = full_dataset.train_test_split(test_size=0.2, seed=42)
train_val = splits["train"].train_test_split(test_size=0.125, seed=42) # 0.125 * 0.8 = 0.1 (10% Val)

datasets = {
    "Train": train_val["train"],
    "Val": train_val["test"],
    "Test": splits["test"]
}

stats_list = []
for split_name, ds in datasets.items():
    cat_counts = Counter()
    for item in ds:
        if "annotations" in item:
            cat_counts.update({ann.get("aspect_category") for ann in item["annotations"]})
    
    for cat in target_categories: 
        stats_list.append({
            "Split": split_name,
            "Category": cat.title(),
            "Count": cat_counts.get(cat, 0)
        })

df_stats = pd.DataFrame(stats_list)

df_pivot = df_stats.pivot(index="Category", columns="Split", values="Count")
df_pivot["Total"] = df_pivot.sum(axis=1)
df_pivot["Distribution (%)"] = (df_pivot["Total"] / df_pivot["Total"].sum() * 100).map("{:.2f}%".format)

print("\n" + "="*50)
print("THỐNG KÊ PHÂN PHỐI DATASET (KIỂM TRA MẤT CÂN BẰNG)")
print("="*50)
display(df_pivot)